In [ ]:
import sys
import cv2
import numpy as np
import tensorflow as tf
import copy

sys.path.append("..")

from src.grid_detection import find_grid_corners, warp_grid
from src.cell_extraction import split_cells
from src.backtracking_solver import solve


In [ ]:
model = tf.keras.models.load_model("../models/digit_classifier.keras")

class_names = [
    "1", "2", "3", "4", "5",
    "6", "7", "8", "9",
    "empty"
]

print("Model loaded")

In [ ]:
CAMERA_INDEX = 0

# Grid detection will only run once every N frames
DETECT_EVERY_N_FRAMES = 10

# If the detected corners move more than this amount,
# consider the Sudoku to have moved.
MOVE_THRESHOLD = 0.03

# Number of failed detections before we unlock the grid
MAX_FAILED_FRAMES = 5

WARP_SIZE = 900

In [ ]:
SEARCHING = "searching for grid"
GRID_LOCKED = "grid locked"
SOLVED = "solved"

state = SEARCHING

In [ ]:
def grid_moved(old_corners, new_corners, frame_shape,
               threshold=MOVE_THRESHOLD):

    if old_corners is None or new_corners is None:
        return True

    old_corners = np.asarray(old_corners, dtype=np.float32)
    new_corners = np.asarray(new_corners, dtype=np.float32)

    movement = np.mean(
        np.linalg.norm(
            old_corners - new_corners,
            axis=1
        )
    )

    h, w = frame_shape[:2]

    diagonal = np.sqrt(h**2 + w**2)

    normalized_movement = movement / diagonal

    return normalized_movement > threshold

In [ ]:
def get_cells(warped):

    gray = cv2.cvtColor(
        warped,
        cv2.COLOR_BGR2GRAY
    )

    return split_cells(gray)

In [ ]:
def predict_digit(cell):

    # Same preprocessing used by your static prototype
    _, cell = cv2.threshold(
        cell,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    cell = cv2.resize(
        cell,
        (28, 28)
    )

    cell = cell.astype("float32") / 255.0

    cell = cell.reshape(
        1,
        28,
        28,
        1
    )

    prediction = model.predict(
        cell,
        verbose=0
    )

    class_id = np.argmax(prediction)
    confidence = np.max(prediction)

    label = class_names[class_id]

    if label == "empty":
        return 0, confidence

    return int(label), confidence

In [ ]:
def recognize_sudoku(warped):

    cells = get_cells(warped)

    grid = []

    for i in range(9):

        row = []

        for j in range(9):

            cell_index = i * 9 + j

            digit, confidence = predict_digit(
                cells[cell_index]
            )

            row.append(digit)

        grid.append(row)

    return grid

In [ ]:
def draw_solution(
    frame,
    corners,
    original_grid,
    solved_grid
):

    if corners is None:
        return frame

    corners = np.asarray(
        corners,
        dtype=np.float32
    )

    # Create the same perspective transform
    _, M = warp_grid(
        frame,
        corners,
        size=WARP_SIZE
    )

    # Create blank image in warped space
    overlay = np.zeros(
        (WARP_SIZE, WARP_SIZE, 3),
        dtype=np.uint8
    )

    cell_size = WARP_SIZE // 9

    for row in range(9):

        for col in range(9):

            # Only draw numbers that were originally empty
            if original_grid[row][col] == 0:

                digit = solved_grid[row][col]

                x = col * cell_size + cell_size // 3
                y = row * cell_size + int(cell_size * 0.7)

                cv2.putText(
                    overlay,
                    str(digit),
                    (x, y),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.3,
                    (0, 255, 0),
                    2,
                    cv2.LINE_AA
                )

    # Convert overlay back to camera perspective
    inverse_M = np.linalg.inv(M)

    h, w = frame.shape[:2]

    projected = cv2.warpPerspective(
        overlay,
        inverse_M,
        (w, h)
    )

    # Only copy non-black pixels
    mask = np.any(
        projected != 0,
        axis=2
    )

    result = frame.copy()

    result[mask] = projected[mask]

    return result

In [ ]:
def draw_solution(
    frame,
    corners,
    original_grid,
    solved_grid
):

    if corners is None:
        return frame

    corners = np.asarray(
        corners,
        dtype=np.float32
    )

    # Create the same perspective transform
    _, M = warp_grid(
        frame,
        corners,
        size=WARP_SIZE
    )

    # Create blank image in warped space
    overlay = np.zeros(
        (WARP_SIZE, WARP_SIZE, 3),
        dtype=np.uint8
    )

    cell_size = WARP_SIZE // 9

    for row in range(9):

        for col in range(9):

            # Only draw numbers that were originally empty
            if original_grid[row][col] == 0:

                digit = solved_grid[row][col]

                x = col * cell_size + cell_size // 3
                y = row * cell_size + int(cell_size * 0.7)

                cv2.putText(
                    overlay,
                    str(digit),
                    (x, y),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.3,
                    (0, 255, 0),
                    2,
                    cv2.LINE_AA
                )

    # Convert overlay back to camera perspective
    inverse_M = np.linalg.inv(M)

    h, w = frame.shape[:2]

    projected = cv2.warpPerspective(
        overlay,
        inverse_M,
        (w, h)
    )

    # Only copy non-black pixels
    mask = np.any(
        projected != 0,
        axis=2
    )

    result = frame.copy()

    result[mask] = projected[mask]

    return result

In [26]:
import cv2
import time

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open camera")

ret, frame = cap.read()

if ret:
    print("Camera working!")
    print("Frame size:", frame.shape)

    cv2.imshow("Camera Test", frame)
    cv2.waitKey(3000)   # show for only 3 seconds
    cv2.destroyAllWindows()
else:
    print("Could not read camera frame")

cap.release()

Camera working!
Frame size: (480, 640, 3)


In [ ]:
import cv2

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Could not open camera.") 

frame_count = 0

last_corners = None
locked_corners = None

original_grid = None
solved_grid = None

failed_detections = 0


while True:

    ret, frame = cap.read()

    if not ret:
        print("Could not read camera frame.")
        break

    frame_count += 1

    display = frame.copy()

    # ==================================================
    # SEARCH / CHECK GRID
    # ==================================================

    check_grid = (
        state == SEARCHING
        or frame_count % DETECT_EVERY_N_FRAMES == 0
    )

    if check_grid:

        try:

            corners = find_grid_corners(frame)

            corners = np.asarray(
                corners,
                dtype=np.float32
            )

            failed_detections = 0

            # ==================================================
            # SEARCHING
            # ==================================================

            if state == SEARCHING:

                print("Grid found.")

                locked_corners = corners.copy()
                last_corners = corners.copy()

                # Warp using your existing function
                warped, M = warp_grid(
                    frame,
                    locked_corners,
                    size=WARP_SIZE
                )

                # ----------------------------------------------
                # CNN ONLY RUNS HERE
                # ----------------------------------------------

                print("Recognizing Sudoku...")

                grid = recognize_sudoku(
                    warped
                )

                print("Detected Sudoku:")

                for row in grid:
                    print(row)

                # ----------------------------------------------
                # Solve using your existing solver
                # ----------------------------------------------

                solved = copy.deepcopy(grid)

                if solve(solved):

                    original_grid = copy.deepcopy(
                        grid
                    )

                    solved_grid = solved

                    state = SOLVED

                    print("Sudoku solved.")

                else:

                    original_grid = copy.deepcopy(
                        grid
                    )

                    solved_grid = None

                    state = GRID_LOCKED

                    print(
                        "Grid locked, "
                        "but Sudoku could not be solved."
                    )

            # ==================================================
            # GRID LOCKED / SOLVED
            # ==================================================

            else:

                # Check whether Sudoku moved
                moved = grid_moved(
                    locked_corners,
                    corners,
                    frame.shape
                )

                if moved:

                    print(
                        "Grid moved. "
                        "Resetting..."
                    )

                    state = SEARCHING

                    locked_corners = None
                    last_corners = None

                    original_grid = None
                    solved_grid = None

                else:

                    # Grid is still in approximately
                    # the same position.
                    last_corners = corners.copy()

        except Exception as e:

            failed_detections += 1

            print(
                f"Grid detection failed: {e}"
            )

            if failed_detections >= MAX_FAILED_FRAMES:

                print("Grid lost.")

                state = SEARCHING

                locked_corners = None
                last_corners = None

                original_grid = None
                solved_grid = None

    # ==================================================
    # DRAW SOLUTION
    # ==================================================

    if (
        state == SOLVED
        and solved_grid is not None
        and original_grid is not None
        and locked_corners is not None
    ):

        display = draw_solution(
            display,
            locked_corners,
            original_grid,
            solved_grid
        )

    # ==================================================
    # DRAW GRID CONTOUR
    # ==================================================

    if locked_corners is not None:

        pts = locked_corners.astype(
            np.int32
        ).reshape(
            (-1, 1, 2)
        )

        cv2.polylines(
            display,
            [pts],
            True,
            (0, 255, 0),
            3
        )

    # ==================================================
    # STATE TEXT
    # ==================================================

    cv2.putText(
        display,
        f"State: {state}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2,
        cv2.LINE_AA
    )

    # ==================================================
    # DISPLAY
    # ==================================================

    cv2.imshow(
        "Realtime Sudoku Solver",
        display
    )

    key = cv2.waitKey(1) & 0xFF

    # Quit
    if key == ord("q"):
        break

    # Reset
    if key == ord("r"):

        print("Manual reset.")

        state = SEARCHING

        locked_corners = None
        last_corners = None

        original_grid = None
        solved_grid = None

        failed_detections = 0


cap.release()
cv2.destroyAllWindows()